# Проверка расчета критериев

Этот notebook проверяет расчет `C1-C4` для выбранного года.

In [4]:
from importlib import import_module
from pathlib import Path
import sys

# Определяем корень проекта.
project_root = Path.cwd()
if not (project_root / "data" / "trade.xlsx").exists():
    project_root = project_root.parent

# Добавляем корень проекта в пути импорта.
sys.path.insert(0, str(project_root))

loader = import_module("src.1_data_loader.loader")
preprocessing = import_module("src.2_preprocessing.preprocessing")
indicators = import_module("src.3_indicators.indicators")

In [5]:
# Год расчета выбирается пользователем.
calculation_year = 2025

raw_data = loader.load_trade_data(project_root / "data" / "trade.xlsx")
prepared_data = preprocessing.preprocess_trade_data(raw_data)
yearly_trade = preprocessing.make_yearly_trade_table(prepared_data)
country_import = preprocessing.make_country_import_table(prepared_data)

indicator_values = indicators.calculate_indicators(
    yearly_trade,
    country_import,
    calculation_year,
)

indicator_values.head(20)

,TNVED,Year,Import,Export,C1,C2,C3,C4
0,841810,2025,2.084822e+08,18947441.75,1.498659e-02,0.000000,0.916689,0.373150
1,841821,2025,4.494547e+07,296556.92,3.230872e-03,0.054670,0.993445,0.636188
2,841829,2025,4.078745e+06,968981.47,2.931976e-04,0.000000,0.808036,0.905747
3,841830,2025,3.347578e+07,2247578.73,2.406382e-03,0.000000,0.937084,0.675522
4,841840,2025,1.886445e+07,1048776.96,1.356057e-03,0.000000,0.947333,0.305240
5,841850,2025,3.413444e+07,6095472.72,2.453729e-03,0.156111,0.848484,0.528235
6,841861,2025,4.227753e+06,43858.43,3.039089e-04,0.072806,0.989732,0.469133
7,841869,2025,8.253162e+07,12423519.11,5.932725e-03,0.000000,0.869164,0.598667
8,841990,2025,1.629126e+08,4874260.67,1.171085e-02,0.622306,0.970950,0.871930
9,842211,2025,1.094512e+08,290074.48,7.867817e-03,0.000000,0.997357,0.577024


In [6]:
# Проверяем базовые свойства результата.
expected_columns = ["TNVED", "Year", "Import", "Export", "C1", "C2", "C3", "C4"]

assert list(indicator_values.columns) == expected_columns
assert set(indicator_values["Year"].unique()) == {calculation_year}
assert indicator_values["Import"].gt(0).all()
assert indicator_values["C1"].between(0, 1).all()
assert indicator_values["C2"].ge(0).all()
assert indicator_values["C3"].between(0, 1).all()
assert indicator_values["C4"].between(0, 1).all()

print("Rows:", len(indicator_values))
print("C1 sum:", indicator_values["C1"].sum())
indicator_values.describe()

Rows: 358
C1 sum: 1.0


,Year,Import,Export,C1,C2,C3,C4
count,358.0,3.580000e+02,3.580000e+02,3.580000e+02,358.000000,358.000000,358.000000
mean,2025.0,3.885824e+07,3.075473e+06,2.793296e-03,0.189490,0.862736,0.666341
std,0.0,1.148853e+08,8.313032e+06,8.258446e-03,0.691762,0.211664,0.227622
min,2025.0,6.389000e+01,0.000000e+00,4.592686e-09,0.000000,0.000141,0.136703
25%,2025.0,1.523087e+06,5.364036e+04,1.094860e-04,0.000000,0.835460,0.478750
50%,2025.0,6.760587e+06,4.705883e+05,4.859799e-04,0.000000,0.950546,0.683867
75%,2025.0,3.304079e+07,2.174049e+06,2.375113e-03,0.058531,0.986134,0.863949
max,2025.0,1.642379e+09,8.036043e+07,1.180612e-01,7.816820,1.000000,1.000000
